In [0]:
%sql
SELECT
    COUNT(*) AS new_subscribers
FROM telecom.gold.dim_subscription
WHERE activation_date IS NOT NULL;

In [0]:
%sql
SELECT
    COUNT(*) AS churned_subscribers
FROM telecom.gold.dim_subscription
WHERE deactivation_date IS NOT NULL;

In [0]:
%sql
SELECT
    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN deactivation_date IS NOT NULL THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS churn_rate_pct
FROM telecom.gold.dim_subscription;

In [0]:
%sql
SELECT
    ROUND(
        AVG(
            (unix_timestamp(call_end_time)
             - unix_timestamp(call_start_time)) / 60.0
        ),
        2
    ) AS average_call_duration_minutes
FROM telecom.gold.fact_call_usage
WHERE call_start_time IS NOT NULL
  AND call_end_time IS NOT NULL;

In [0]:
# ============================================================
# TELECOM KPI SUMMARY
# ============================================================

kpi_summary = spark.sql("""
SELECT

    (
        SELECT COUNT(*)
        FROM telecom.gold.dim_subscription
        WHERE activation_date IS NOT NULL
    ) AS new_subscribers,

    (
        SELECT COUNT(*)
        FROM telecom.gold.dim_subscription
        WHERE deactivation_date IS NOT NULL
    ) AS churned_subscribers,

    (
        SELECT ROUND(
            100.0 *
            SUM(
                CASE
                    WHEN deactivation_date IS NOT NULL THEN 1
                    ELSE 0
                END
            ) / COUNT(*),
            2
        )
        FROM telecom.gold.dim_subscription
    ) AS churn_rate_pct,

    (
        SELECT ROUND(
            AVG(
                (
                    unix_timestamp(call_end_time)
                    - unix_timestamp(call_start_time)
                ) / 60.0
            ),
            2
        )
        FROM telecom.gold.fact_call_usage
        WHERE call_start_time IS NOT NULL
          AND call_end_time IS NOT NULL
    ) AS average_call_duration_minutes,

    (
        SELECT ROUND(
            total_collected_revenue /
            NULLIF(total_customers, 0),
            2
        )
        FROM telecom.gold.vw_executive_kpis
    ) AS arpu

""")

display(kpi_summary)

In [0]:
display(
    spark.sql("""
        DESCRIBE telecom.gold.fact_data_usage
    """)
)

In [0]:
# ============================================================
# AVERAGE DATA USAGE PER CUSTOMER
# ============================================================

avg_data_usage = spark.sql("""
    SELECT
        ROUND(
            SUM(data_consumed_mb) / 1024.0
            / COUNT(DISTINCT customer_id),
            2
        ) AS average_data_usage_gb_per_customer
    FROM telecom.gold.fact_data_usage
""")

display(avg_data_usage)

In [0]:
# ============================================================
# TELECOM KPI SUMMARY
# ============================================================

kpi_summary = spark.sql("""
SELECT

    -- New Subscribers
    (
        SELECT COUNT(*)
        FROM telecom.gold.dim_subscription
        WHERE activation_date IS NOT NULL
    ) AS new_subscribers,

    -- Churned Subscribers
    (
        SELECT COUNT(*)
        FROM telecom.gold.dim_subscription
        WHERE deactivation_date IS NOT NULL
    ) AS churned_subscribers,

    -- Churn Rate
    (
        SELECT ROUND(
            100.0 *
            SUM(
                CASE
                    WHEN deactivation_date IS NOT NULL THEN 1
                    ELSE 0
                END
            ) / COUNT(*),
            2
        )
        FROM telecom.gold.dim_subscription
    ) AS churn_rate_pct,

    -- Average Call Duration
    (
        SELECT ROUND(
            AVG(
                (
                    unix_timestamp(call_end_time)
                    - unix_timestamp(call_start_time)
                ) / 60.0
            ),
            2
        )
        FROM telecom.gold.fact_call_usage
        WHERE call_start_time IS NOT NULL
          AND call_end_time IS NOT NULL
    ) AS average_call_duration_minutes,

    -- Average Data Usage per Customer
    (
        SELECT ROUND(
            SUM(data_consumed_mb) / 1024.0
            / COUNT(DISTINCT customer_id),
            2
        )
        FROM telecom.gold.fact_data_usage
    ) AS average_data_usage_gb_per_customer,

    -- ARPU
    (
        SELECT ROUND(
            total_collected_revenue /
            NULLIF(total_customers, 0),
            2
        )
        FROM telecom.gold.vw_executive_kpis
    ) AS arpu,

    -- Payment Success Rate
    (
        SELECT ROUND(
            100.0 *
            SUM(successful_payment_count) /
            NULLIF(SUM(total_payments), 0),
            2
        )
        FROM telecom.gold.customer_360
    ) AS payment_success_rate_pct,

    -- Customer Retention Rate
    (
        SELECT ROUND(
            100.0 *
            SUM(
                CASE
                    WHEN cancelled_subscriptions = 0 THEN 1
                    ELSE 0
                END
            ) / COUNT(*),
            2
        )
        FROM telecom.gold.customer_360
    ) AS customer_retention_rate_pct

""")

display(kpi_summary)